In [27]:
import pandas as pd
import numpy as np 
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder 
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

In [28]:
# Load the dataset 
df = pd.read_csv("Churn_Modelling.csv")
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [29]:
# Preprocess the data 
# Drop irrelevant columns 
df = df.drop(['RowNumber',"CustomerId","Surname"], axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [30]:
# Encode Categorical Variables 
label_encoder_gender = LabelEncoder() 
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.head(2)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0


In [31]:
# Onehot encode Geography 
onehot_encoder_geo = OneHotEncoder(sparse_output=False) 
geo_encoder = onehot_encoder_geo.fit_transform(df[['Geography']])

encoded_geo_df = pd.DataFrame(
    geo_encoder,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']),
    index=df.index
)

df = pd.concat([df.drop(columns=['Geography']), encoded_geo_df], axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [32]:
# Save the encoders  
with open('label_encoder_gender.pkl', 'wb') as file: 
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl', 'wb') as file: 
    pickle.dump(onehot_encoder_geo, file)

In [33]:
# divide the dataset into dependent and independent features 
X = df.drop('Exited', axis=1)
y = df['Exited']

# Split the Data into train test split 
X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42)

In [34]:
# Scale the features 
Scaler = StandardScaler() 
X_train = Scaler.fit_transform(X_train)
X_test = Scaler.transform(X_test)

In [35]:
# Save the scaler
with open('Scaler.pkl', 'wb') as file: 
    pickle.dump(Scaler, file)

# ANN Implementation

In [36]:
# lets make Custom dataset class 

class MyCustDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels.to_numpy(), dtype = torch.float32).view(-1,1)

    def __len__(self):
        return len(self.features) 

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [37]:
# Create  tain dataset and test dataset objects. 
train_dataset = MyCustDataset(X_train,y_train) 
test_dataset = MyCustDataset(X_test, y_test) 

In [38]:
# Create the train and test loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [39]:
# Define Model

class MyNN(nn.Module): 
    
    def __init__(self, num_freatures): 

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_freatures,64), 
            nn.BatchNorm1d(64), 
            nn.ReLU(),
            nn.Dropout(0.3), 
            nn.Linear(64,32), 
            nn.BatchNorm1d(32), 
            nn.ReLU(), 
            nn.Dropout(0.3), 
            nn.Linear(32,1),
            nn.Sigmoid()
        )
    
    def forward(self, x): 
        return self.model(x)


In [40]:
# set learning rate and the epochs 
learning_rate = 0.01 
epochs = 100

In [41]:
# Initiate the model 
model = MyNN(X_train.shape[1])

# Loss Function 
criterion = nn.BCELoss() 

# Optimizer 
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

# writer 
writer = SummaryWriter("runs/churn_prediction")

In [42]:
# Training loop 
best_loss = float('inf') 
patience = 5
counter = 0

for epoch in range(epochs): 

    total_epoch_loss = 0 

    for batch_features, batch_labels in train_loader: 

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad() 

        loss.backward()

        optimizer.step() 

        total_epoch_loss = total_epoch_loss + loss.item() 

    avg_loss = total_epoch_loss/len(train_loader) 
    print(f'Epoch : {epoch + 1} , Loss : {avg_loss}')

    # early stopping check 
    if avg_loss < best_loss: 
        best_loss = avg_loss
        counter = 0 
        torch.save({              
            "model_state": model.state_dict(),
            "num_features": X_train.shape[1]
        }, "best_model.pth")
    else:
        counter += 1               # add 1 bad epoch
        if counter >= patience:    # if 10 bad epochs in a row
            print(f"⛔ Stopped at epoch {epoch+1}!")
            break                  # stop training

Epoch : 1 , Loss : 0.43766874259710314
Epoch : 2 , Loss : 0.3895902438759804
Epoch : 3 , Loss : 0.38062769931554796
Epoch : 4 , Loss : 0.3747520915865898
Epoch : 5 , Loss : 0.3695613366365433
Epoch : 6 , Loss : 0.3697013739347458
Epoch : 7 , Loss : 0.36828072047233584
Epoch : 8 , Loss : 0.3718238055706024
Epoch : 9 , Loss : 0.3693368881344795
Epoch : 10 , Loss : 0.36580313158035277
Epoch : 11 , Loss : 0.36187022072076797
Epoch : 12 , Loss : 0.3656206278800964
Epoch : 13 , Loss : 0.36104268008470536
Epoch : 14 , Loss : 0.3669666958451271
Epoch : 15 , Loss : 0.36188584876060487
Epoch : 16 , Loss : 0.36084606289863586
Epoch : 17 , Loss : 0.3654135545492172
Epoch : 18 , Loss : 0.36074534207582476
Epoch : 19 , Loss : 0.35680499517917635
Epoch : 20 , Loss : 0.35833087837696076
Epoch : 21 , Loss : 0.36092908841371535
Epoch : 22 , Loss : 0.3625142970085144
Epoch : 23 , Loss : 0.3620187491774559
Epoch : 24 , Loss : 0.3574147778749466
⛔ Stopped at epoch 24!


In [43]:
# Log the training loss
writer.add_scalar("Training Loss", loss.item(), epoch)

In [44]:
# model evaluation 
correct = 0 
total = 0 

model.eval() 

with torch.no_grad(): 

    for batch_features, batch_labels in test_loader:  
        outputs = model(batch_features) 
        predicted = (outputs >= 0.5).float() 
        total += batch_labels.size(0) 
        correct += (predicted == batch_labels).sum().item() 

accuracy = correct/total * 100 
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 85.90%


In [45]:
# log the accuracy 
writer.add_scalar("Validation Accuracy",accuracy,epoch)

# Close the writer
writer.close()

In [46]:
# save the model
torch.save({
    "model_state": model.state_dict(),
    "num_features": 12
}, "model.pth")